# Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import shapiro

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import seasonal_decompose

# Load Data

In [ ]:
df_jatim_raw = pd.read_csv("dataset/Data_Jatim.csv")
df_jatim_raw["Tanggal"] = pd.to_datetime(df_jatim_raw["Tanggal"])
df_jatim_raw["Inflasi"] = pd.to_numeric(df_jatim_raw["Inflasi"], errors="coerce")
display(df_jatim_raw.head())

# Data Information

In [ ]:
df_jatim = (df_jatim_raw.groupby("Tanggal")["Inflasi"]
            .mean().reset_index()
            .rename(columns={
                "Tanggal": "tanggal",
                "Inflasi": "inflasi"
            }).set_index("tanggal").sort_index().asfreq("MS"))

df_jatim = df_jatim.loc["2015-01-01":"2025-12-01"]
display(df_jatim.head())

# Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df_jatim.index, df_jatim["inflasi"], label="Inflasi")
plt.title("Inflasi Bulanan Jawa Timur (2015–2025)")
plt.xlabel("Tahun")
plt.ylabel("Inflasi M-to-M (%)")
plt.grid(True)
plt.legend()
plt.show()

## Rolling Mean and Standard Deviation

In [ ]:
rolling_mean = df_jatim["inflasi"].rolling(window=12).mean()
rolling_std = df_jatim["inflasi"].rolling(window=12).std()

plt.figure(figsize=(12, 5))
plt.plot(df_jatim.index, df_jatim["inflasi"], label="Data Aktual")
plt.plot(rolling_mean.index, rolling_mean, label="Rolling Mean 12 Bulan")
plt.plot(rolling_std.index, rolling_std, label="Rolling Std 12 Bulan")

plt.title("Rolling Mean dan Rolling Standard Deviation Inflasi Jawa Timur")
plt.xlabel("Tahun")
plt.ylabel("Inflasi M-to-M (%)")
plt.legend()
plt.grid(True)
plt.show()

## Seasonal Decomposition

In [ ]:
decomposition = seasonal_decompose(
    df_jatim["inflasi"],
    model="additive",
    period=12
)

fig = decomposition.plot()
fig.set_size_inches(12, 8)
plt.show()

## Train-Test Split

In [ ]:
y = df_jatim["inflasi"]
df_train = y.loc["2015-01-01":"2024-12-01"]
df_test = y.loc["2025-01-01":"2025-12-01"]

print("Jumlah train:", len(df_train))
print("Jumlah test:", len(df_test))
print("Periode train:", df_train.index.min(), "-", df_train.index.max())
print("Periode test:", df_test.index.min(), "-", df_test.index.max())

# Stationarity Test

In [ ]:
def adf_test(data, name="Data"):
    result = adfuller(data.dropna())
    print(f"ADF Test - {name}")
    print(f"ADF Statistic: {result[0]}")
    print(f"p-value: {result[1]}")
    
    print("Critical Values:")
    for key, value in result[4].items():
        print(f"{key}: {value}")
    
    if result[1] < 0.05:
        print("Kesimpulan: stasioner")
    else:
        print("Kesimpulan: belum stasioner")
    print("-" * 60)

adf_test(df_train, "Data Train")
adf_test(df_train.diff().dropna(), "Differencing Regular d=1")
adf_test(df_train.diff(12).dropna(), "Differencing Musiman D=1")
adf_test(df_train.diff().diff(12).dropna(), "Differencing Regular d=1 dan Musiman D=1")

# ACF and PACF

In [ ]:
train = df_train.dropna()
# train_d1 = df_train.diff().dropna()
# train_D1 = df_train.diff(12).dropna()
# train_d1_D1 = df_train.diff().diff(12).dropna()

In [ ]:
def plot_acf_pacf(series, title, lags=36):
    plt.figure(figsize=(10, 4))
    plot_acf(series, lags=lags)
    plt.title(f"ACF - {title}")
    plt.xlabel("Lag")
    plt.ylabel("Autocorrelation")
    plt.show()

    plt.figure(figsize=(10, 4))
    plot_pacf(series, lags=lags, method="ywm")
    plt.title(f"PACF - {title}")
    plt.xlabel("Lag")
    plt.ylabel("Partial Autocorrelation")
    plt.show()

In [ ]:
plot_acf_pacf(train, "Data Train", lags=36)
# plot_acf_pacf(train_d1, "Differencing Regular d=1", lags=36)
# plot_acf_pacf(train_D1, "Differencing Musiman D=1", lags=36)
# plot_acf_pacf(train_d1_D1, "Differencing Regular d=1 dan Musiman D=1", lags=36)

# SARIMA Model Estimation

In [ ]:
s = 12

results = []
failed_models = 0

p_values = range(0, 3)
d_values = range(0, 2)
q_values = range(0, 3)

P_values = range(0, 2)
D_values = range(0, 2)
Q_values = range(0, 2)

for p in p_values:
    for d in d_values:
        for q in q_values:
            for P in P_values:
                for D in D_values:
                    for Q in Q_values:
                        order = (p, d, q)
                        seasonal_order = (P, D, Q, s)
                        if order == (0, 0, 0) and seasonal_order == (0, 0, 0, s):
                            continue

                        try:
                            model = SARIMAX(
                                df_train,
                                order=order,
                                seasonal_order=seasonal_order,
                                enforce_stationarity=False,
                                enforce_invertibility=False
                            )

                            fitted = model.fit(disp=False, maxiter=300)
                            forecast_test = fitted.get_forecast(steps=len(df_test))
                            pred_test = forecast_test.predicted_mean
                            pred_test.index = df_test.index

                            rmse = np.sqrt(mean_squared_error(df_test, pred_test))
                            mae = mean_absolute_error(df_test, pred_test)
                            residual = fitted.resid.dropna()
                            lb_test = acorr_ljungbox(
                                residual,
                                lags=[12],
                                return_df=True
                            )

                            lb_pvalue = lb_test["lb_pvalue"].iloc[0]
                            try:
                                shapiro_pvalue = shapiro(residual).pvalue
                            except:
                                shapiro_pvalue = np.nan
                            results.append({
                                "order": order,
                                "seasonal_order": seasonal_order,
                                "aic": fitted.aic,
                                "bic": fitted.bic,
                                "rmse": rmse,
                                "mae": mae,
                                "ljungbox_pvalue": lb_pvalue,
                                "shapiro_pvalue": shapiro_pvalue
                            })
                        except:
                            failed_models += 1
                            continue

df_results = pd.DataFrame(results)
print("Jumlah model berhasil:", len(df_results))
print("Jumlah model gagal:", failed_models)

if len(df_results) > 0:
    df_results = df_results.sort_values(["aic", "rmse"]).reset_index(drop=True)
    print("Top model berdasarkan AIC:")
    display(df_results.head(10))
else:
    print("Tidak ada model yang berhasil diestimasi.")

In [ ]:
df_results.to_csv("top_model_sarima_jatim.csv", index=False)

models = df_results[
    (df_results["ljungbox_pvalue"] > 0.05) & (df_results["shapiro_pvalue"] > 0.05)
].copy()

print(f"Jumlah model yang lolos diagnostic checking: {len(models)}")
if len(models) > 0:
    best_model = models.sort_values(["rmse", "aic"]).iloc[0]
else:
    best_model = df_results.iloc[0]

print("Model SARIMA terbaik")
display(best_model)

# Fit Model

In [ ]:
best_order = best_model["order"]
best_seasonal_order = best_model["seasonal_order"]

best_model_train = SARIMAX(
    df_train,
    order=best_order,
    seasonal_order=best_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)

best_fitted_train = best_model_train.fit(disp=False, maxiter=300)
print(best_fitted_train.summary())

# Evaluasi out-sample

In [ ]:
forecast_test = best_fitted_train.get_forecast(steps=len(df_test))
pred_test = forecast_test.predicted_mean
pred_test.index = df_test.index

df_eval = pd.DataFrame({
    "actual": df_test,
    "forecast": pred_test
})
df_eval["error"] = df_eval["actual"] - df_eval["forecast"]
df_eval["abs_error"] = df_eval["error"].abs()
df_eval["squared_error"] = df_eval["error"] ** 2
rmse = np.sqrt(mean_squared_error(df_eval["actual"], df_eval["forecast"]))
mae = mean_absolute_error(df_eval["actual"], df_eval["forecast"])

display(df_eval)
print("RMSE:", rmse)
print("MAE :", mae)

df_eval.to_csv("evaluasi_outsample_2025_jatim.csv")

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(df_train.index, df_train, label="Train")
plt.plot(df_test.index, df_test, label="Test")
plt.plot(pred_test.index, pred_test, label="Forecast 2025")

plt.title("Evaluasi Forecast Inflasi Bulanan Jawa Timur 2025")
plt.xlabel("Tahun")
plt.ylabel("Inflasi M-to-M (%)")
plt.legend()
plt.grid(True)
plt.show()

# Diagnostic Checking

In [ ]:
residual = best_fitted_train.resid.dropna()

plt.figure(figsize=(12, 4))
plt.plot(residual)
plt.title("Residual Model SARIMA Terbaik")
plt.xlabel("Tahun")
plt.ylabel("Residual")
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))
plot_acf(residual, lags=36)
plt.title("ACF Residual Model SARIMA Terbaik")
plt.show()

lb = acorr_ljungbox(residual, lags=[12, 24], return_df=True)
print("Ljung-Box Test:")
print(lb)

shapiro_stat, shapiro_p = shapiro(residual)
print("Shapiro-Wilk Test")
print("Statistic:", shapiro_stat)
print("p-value  :", shapiro_p)

# Forecasting

In [ ]:
final_model = SARIMAX(
    y,
    order=best_order,
    seasonal_order=best_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)

final_fitted = final_model.fit(disp=False, maxiter=300)

forecast_2026 = final_fitted.get_forecast(steps=12)
forecast_mean = forecast_2026.predicted_mean
forecast_ci = forecast_2026.conf_int(alpha=0.05)

df_forecast = pd.DataFrame({
    "forecast": forecast_mean,
    "lower_95": forecast_ci.iloc[:, 0],
    "upper_95": forecast_ci.iloc[:, 1]
})

display(df_forecast)
df_forecast.to_csv("forecast_inflasi_2026_sarima_jatim.csv")

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(y.index, y, label="Data Historis 2015–2025")
plt.plot(df_forecast.index, df_forecast["forecast"], label="Forecast 2026")

plt.fill_between(
    df_forecast.index,
    df_forecast["lower_95"],
    df_forecast["upper_95"],
    alpha=0.2,
    label="Interval Prediksi 95%"
)

plt.title("Forecast Inflasi Bulanan Jawa Timur Tahun 2026")
plt.xlabel("Tahun")
plt.ylabel("Inflasi M-to-M (%)")
plt.legend()
plt.grid(True)
plt.show()